In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import fastdtw as dtw
import pandas as pd
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks

Lets start by importing and visualizing the dataset:

In [10]:
# Load the original dataset to get shape
df = pd.read_csv('exo_data.csv')

Next, lets extract some features for the pre-DTW search filter. We'll start by defining the extraction function.

In [11]:
def extract_features(row):
    flux_values = row[1:]  # Assuming LABEL is the first column
    N = len(flux_values)

    # Basic stats
    mean_val = np.mean(flux_values)
    median_val = np.median(flux_values)
    min_val = np.min(flux_values)
    max_val = np.max(flux_values)
    std_val = np.std(flux_values)
    skew_val = skew(flux_values)
    kurt_val = kurtosis(flux_values)
    energy_val = np.sum(np.square(flux_values))
    peaks, _ = find_peaks(flux_values)
    troughs, _ = find_peaks(-flux_values)

    # === FFT Features ===
    fft_vals = np.fft.fft(flux_values)
    fft_magnitudes = np.abs(fft_vals)[:N // 2]  # Keep only the first half (positive freqs)
    fft_magnitudes[0] = 0  # Zero out DC component

    # Get indices of top 3 dominant frequency bins
    dominant_indices = np.argsort(fft_magnitudes)[-3:][::-1]

    return pd.Series({
        'mean': mean_val,
        'median': median_val,
        'min': min_val,
        'max': max_val,
        'std': std_val,
        'skew': skew_val,
        'kurtosis': kurt_val,
        'num_peaks': len(peaks),
        'num_troughs': len(troughs),
        'dominant_freq_1': dominant_indices[0],
        'dominant_freq_2': dominant_indices[1],
        'dominant_freq_3': dominant_indices[2]
    })


Now lets actually apply this to our dataframe and see the result:

In [12]:
# Separate label
labels = df["LABEL"]

# Apply the feature extraction
features_df = df.drop(columns=["LABEL"]).apply(extract_features, axis=1)

# Add back labels
features_df["LABEL"] = labels

Next, as a sanity check lets train a baseline KNN model with k=5 and see how this model performs; this should be a good proxy for measuring how meaningful our features really are.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Separate features and labels
X = features_df.drop(columns=['LABEL'])
y = features_df['LABEL']

# Train-test split (80-20) for features_df
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Ensure that df follows the same split (we drop the 'LABEL' column from df)
df_train = df.iloc[X_train.index].drop(columns=['LABEL'])  # Using the same index for training set
df_test = df.iloc[X_test.index].drop(columns=['LABEL'])    # Using the same index for test set

# Standardize features (very important for KNN!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize KNN with k=5
knn = KNeighborsClassifier(n_neighbors=20, weights='distance', metric='manhattan')
knn.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = knn.predict(X_test_scaled)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9378698224852071

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.97      0.97      1204
           1       0.75      0.65      0.70       148

    accuracy                           0.94      1352
   macro avg       0.85      0.81      0.83      1352
weighted avg       0.93      0.94      0.94      1352



Now  lets create the DTW integrated hierarchical classifier

In [19]:
from fastdtw import fastdtw
from scipy.spatial.distance import cityblock  # Manhattan distance
from tqdm import tqdm

class HierarchicalKNNDTWClassifier:
    def __init__(self, k=10, metric='manhattan'):
        self.k = k
        self.metric = cityblock if metric == 'manhattan' else None  # You can extend this
        self.X_feat = None
        self.X_raw = None
        self.y = None

    def fit(self, X_feat, X_raw, y):
        self.X_feat = X_feat
        self.X_raw = X_raw
        self.y = y.reset_index(drop=True)

    def predict(self, X_feat_test, X_raw_test):
        preds = []

        for i in tqdm(range(len(X_feat_test)), desc="Classifying with DTW", unit="sample"):
            # Step 1: Find top-k neighbors using L1 distance in feature space
            dists = [self.metric(X_feat_test.iloc[i], train_vec) for train_vec in self.X_feat.values]
            top_k_idx = np.argsort(dists)[:self.k]

            # Step 2: Among those, compute DTW distances
            dtw_dists = []
            test_series = X_raw_test.iloc[i].values

            for idx in top_k_idx:
                train_series = self.X_raw.iloc[idx].values
                distance, _ = fastdtw(test_series, train_series)
                dtw_dists.append((distance, self.y.iloc[idx]))

            # Step 3: Pick label of the closest DTW series
            dtw_dists.sort()
            preds.append(dtw_dists[0][1])  # Label of smallest DTW

        return preds

Lets implement and evaluate the hierarchical model

In [ ]:
model = HierarchicalKNNDTWClassifier(k=10)
model.fit(X_train, df_train, y_train)

# Predict on test set
y_pred = model.predict(X_test, df_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Classifying with DTW:  81%|████████▏ | 1099/1352 [50:18<10:19,  2.45s/sample]